# Spatio-Temporal Mobility Demand Forecasting
## Final Solution (Epitome)

**Objective**: Accurately predict mobility demand across a city grid (Geohashes) over time.

This notebook encapsulates our entire final pipeline (V7). The core challenges of this dataset are:
1. **Zero-Inflated, Right-Skewed Demand**: Demand frequently hits exactly 0.0, especially at night.
2. **Spatial Topology**: Geohashes are physically connected, meaning demand spills over into neighboring areas.
3. **Auto-Regressive Sequencing**: We must predict step $T$, and use that prediction to forecast step $T+1$ (an AR loop).
4. **Target Leakage Trap**: Standard mean encodings will cause the model to overfit on the training data. We use strict Out-of-Fold (OOF) target encodings to prevent this.

Steps covered in this notebook:
1. Setup and Seed Initialization
2. Utility Functions (Geohash Decoding & Fourier Harmonics)
3. Strict OOF Target Encodings & Historical Stats
4. Spatial Graph Features (Centrality & Neighborhood Spillover)
5. Custom Asymmetric Loss Functions
6. Auto-Regressive Ensemble Training (CatBoost, LightGBM, XGBoost)


In [ ]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

---
## 1. Time & Space Utilities
We decode base-32 Geohashes into approximate numerical `latitude` and `longitude`.
We also use Fourier Harmonics (Sine/Cosine waves) to encode the cyclicity of a 24-hour day.

In [ ]:
def decode_geohash(geohash):
    BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"
    base32_map = {char: i for i, char in enumerate(BASE32)}
    lat_interval = (-90.0, 90.0)
    lon_interval = (-180.0, 180.0)
    is_even = True
    for char in geohash:
        val = base32_map[char]
        for mask in [16, 8, 4, 2, 1]:
            bit = 1 if (val & mask) else 0
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2.0
                if bit == 1:
                    lon_interval = (mid, lon_interval[1])
                else:
                    lon_interval = (lon_interval[0], mid)
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2.0
                if bit == 1:
                    lat_interval = (mid, lat_interval[1])
                else:
                    lat_interval = (lat_interval[0], mid)
            is_even = not is_even
    lat = (lat_interval[0] + lat_interval[1]) / 2.0
    lon = (lon_interval[0] + lon_interval[1]) / 2.0
    return lat, lon

def encode_fourier_harmonics(df, minutes_col, cycles=[1440, 720, 480, 360]):
    df_copy = df.copy()
    for cycle in cycles:
        angle = 2.0 * np.pi * df_copy[minutes_col] / cycle
        df_copy[f'sin_{cycle}'] = np.sin(angle)
        df_copy[f'cos_{cycle}'] = np.cos(angle)
    return df_copy

---
## 2. Leakage-Free Target Encodings
Gradient Boosted Trees will "cheat" if they see the target value during training. 
To extract features like the *historical average demand of a specific geohash*, we use **K-Fold Out-of-Fold (OOF) Target Encoding**. The training data is split into 5 folds. The average demand for a geohash in Fold 1 is computed strictly using data from Folds 2-5.

In [ ]:
def add_oof_target_encodings(train_df, test_df, col, target_col, n_splits=5, smoothing_val=10.0):
    train_copy = train_df.copy()
    test_copy = test_df.copy()
    
    train_copy[f'{col}_te'] = np.nan
    test_copy[f'{col}_te'] = np.nan
    
    global_mean = train_copy[target_col].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for train_idx, val_idx in kf.split(train_copy):
        fold_train = train_copy.iloc[train_idx]
        stats = fold_train.groupby(col)[target_col].agg(['count', 'mean'])
        encoded_map = (stats['count'] * stats['mean'] + smoothing_val * global_mean) / (stats['count'] + smoothing_val)
        encoded_dict = encoded_map.to_dict()
        train_copy.iloc[val_idx, train_copy.columns.get_loc(f'{col}_te')] = train_copy.iloc[val_idx][col].map(encoded_dict).fillna(global_mean)
        
    stats_full = train_copy.groupby(col)[target_col].agg(['count', 'mean'])
    encoded_map_full = (stats_full['count'] * stats_full['mean'] + smoothing_val * global_mean) / (stats_full['count'] + smoothing_val)
    encoded_dict_full = encoded_map_full.to_dict()
    
    test_copy[f'{col}_te'] = test_copy[col].map(encoded_dict_full).fillna(global_mean)
    train_copy[f'{col}_te'] = train_copy[f'{col}_te'].fillna(global_mean)
    
    return train_copy, test_copy

def add_oof_historical_geohash_stats(train_df, test_df, n_splits=5):
    train_copy = train_df.copy()
    test_copy = test_df.copy()
    
    cols_to_add = ['geohash_mean_demand', 'geohash_std_demand', 'geohash_max_demand', 'geohash_spike_rate']
    for col in cols_to_add:
        train_copy[col] = np.nan
        test_copy[col] = np.nan
        
    train_copy['is_spike'] = (train_copy['demand'] == 1.0).astype(int)
    
    global_mean = train_copy['demand'].mean()
    global_std = train_copy['demand'].std()
    global_max = train_copy['demand'].max()
    global_spike_rate = train_copy['is_spike'].mean()
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for train_idx, val_idx in kf.split(train_copy):
        fold_train = train_copy.iloc[train_idx]
        stats = fold_train.groupby('geohash')['demand'].agg(['mean', 'std', 'max']).reset_index()
        stats.columns = ['geohash', 'mean_val', 'std_val', 'max_val']
        spike_stats = fold_train.groupby('geohash')['is_spike'].mean().reset_index()
        spike_stats.columns = ['geohash', 'spike_rate_val']
        stats = stats.merge(spike_stats, on='geohash')
        stats['std_val'] = stats['std_val'].fillna(0.0)
        
        val_fold = train_copy.iloc[val_idx][['geohash']].merge(stats, on='geohash', how='left')
        val_fold['mean_val'] = val_fold['mean_val'].fillna(global_mean)
        val_fold['std_val'] = val_fold['std_val'].fillna(global_std)
        val_fold['max_val'] = val_fold['max_val'].fillna(global_max)
        val_fold['spike_rate_val'] = val_fold['spike_rate_val'].fillna(global_spike_rate)
        
        train_copy.iloc[val_idx, train_copy.columns.get_loc('geohash_mean_demand')] = val_fold['mean_val'].values
        train_copy.iloc[val_idx, train_copy.columns.get_loc('geohash_std_demand')] = val_fold['std_val'].values
        train_copy.iloc[val_idx, train_copy.columns.get_loc('geohash_max_demand')] = val_fold['max_val'].values
        train_copy.iloc[val_idx, train_copy.columns.get_loc('geohash_spike_rate')] = val_fold['spike_rate_val'].values
        
    stats_full = train_copy.groupby('geohash')['demand'].agg(['mean', 'std', 'max']).reset_index()
    stats_full.columns = ['geohash', 'mean_val', 'std_val', 'max_val']
    spike_stats_full = train_copy.groupby('geohash')['is_spike'].mean().reset_index()
    spike_stats_full.columns = ['geohash', 'spike_rate_val']
    stats_full = stats_full.merge(spike_stats_full, on='geohash')
    stats_full['std_val'] = stats_full['std_val'].fillna(0.0)
    
    test_fold = test_copy[['geohash']].merge(stats_full, on='geohash', how='left')
    test_copy['geohash_mean_demand'] = test_fold['mean_val'].fillna(global_mean).values
    test_copy['geohash_std_demand'] = test_fold['std_val'].fillna(global_std).values
    test_copy['geohash_max_demand'] = test_fold['max_val'].fillna(global_max).values
    test_copy['geohash_spike_rate'] = test_fold['spike_rate_val'].fillna(global_spike_rate).values
    
    # Also add geohash_time_block_spike_rate
    train_copy['geohash_time_block_spike_rate'] = np.nan
    test_copy['geohash_time_block_spike_rate'] = np.nan
    for train_idx, val_idx in kf.split(train_copy):
        fold_train = train_copy.iloc[train_idx]
        tb_spike_stats = fold_train.groupby('geohash_time_block')['is_spike'].mean().to_dict()
        train_copy.iloc[val_idx, train_copy.columns.get_loc('geohash_time_block_spike_rate')] = train_copy.iloc[val_idx]['geohash_time_block'].map(tb_spike_stats).fillna(global_spike_rate)
        
    tb_spike_stats_full = train_copy.groupby('geohash_time_block')['is_spike'].mean().to_dict()
    test_copy['geohash_time_block_spike_rate'] = test_copy['geohash_time_block'].map(tb_spike_stats_full).fillna(global_spike_rate)
    
    train_copy = train_copy.drop(columns=['is_spike'])
    return train_copy, test_copy

---
## 3. Spatial Centrality
We compute spatial centrality using inverse distances to $k$-nearest neighbors (for $k=3, 10, 20$). Dense urban centers will have a high centrality, while isolated suburban areas will have a low centrality. This allows the model to differentiate between structural spatial properties.

In [ ]:
def compute_multiscale_spatial_centrality(train_df, test_df, k_values=[3, 10, 20]):
    train_copy = train_df.copy()
    test_copy = test_df.copy()
    unique_ghs = pd.concat([train_copy['geohash'], test_copy['geohash']]).unique()
    coord_map = {gh: decode_geohash(gh) for gh in unique_ghs}
    for df in [train_copy, test_copy]:
        df['latitude'] = df['geohash'].map(lambda x: coord_map[x][0])
        df['longitude'] = df['geohash'].map(lambda x: coord_map[x][1])
    train_unique_gh = train_copy[['geohash', 'latitude', 'longitude']].drop_duplicates().reset_index(drop=True)
    coords_array = train_unique_gh[['latitude', 'longitude']].values
    for k in k_values:
        n_neighbors = min(k + 1, len(coords_array))
        if n_neighbors <= 1:
            train_copy[f'spatial_centrality_{k}'] = 1.0
            test_copy[f'spatial_centrality_{k}'] = 1.0
            continue
        nn = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean')
        nn.fit(coords_array)
        distances, _ = nn.kneighbors(coords_array)
        avg_dist = distances[:, 1:].mean(axis=1)
        epsilon = 1e-5
        train_unique_gh[f'spatial_centrality_{k}'] = 1.0 / (avg_dist + epsilon)
        centrality_map = train_unique_gh.set_index('geohash')[f'spatial_centrality_{k}'].to_dict()
        global_avg = train_unique_gh[f'spatial_centrality_{k}'].mean()
        train_copy[f'spatial_centrality_{k}'] = train_copy['geohash'].map(centrality_map).fillna(global_avg)
        test_copy[f'spatial_centrality_{k}'] = test_copy['geohash'].map(centrality_map).fillna(global_avg)
    return train_copy, test_copy

---
## 4. Master Feature Engineering Pipeline
This function integrates all time-series lag features (e.g., `demand_lag1`, `ewma_03`), spatial spillover variables, and our OOF estimators into one cohesive pipeline.

In [ ]:
def pipeline_feature_engineering(train, test):
    print("Starting pipeline feature engineering...")
    train_clean = train.copy()
    test_clean = test.copy()
    
    train_clean['idle_capacity'] = 1.0 - train_clean['demand']
    train_clean['log1p_demand'] = np.log1p(train_clean['demand'])
    
    for df in [train_clean, test_clean]:
        df['interval_key'] = df['day'].astype(str) + '_' + df['timestamp'].astype(str)
        df['minutes'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]) * 60 + int(x.split(':')[1]))
        df['day_of_week'] = (df['day'] - 1) % 7
        df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
        df['time_block'] = df['minutes'] // 180
        df['time_block_1h'] = df['minutes'] // 60
        df['geohash_time_block'] = df['geohash'].astype(str) + '_' + df['time_block'].astype(str)
        df['geohash_time_block_1h'] = df['geohash'].astype(str) + '_' + df['time_block_1h'].astype(str)
        df['geohash_weather'] = df['geohash'].astype(str) + '_' + df['Weather'].astype(str)
        
    train_clean = encode_fourier_harmonics(train_clean, 'minutes')
    test_clean = encode_fourier_harmonics(test_clean, 'minutes')
    
    print("Computing cross-sectional features...")
    combined_temp = pd.concat([train_clean, test_clean], ignore_index=True)
    combined_temp['interval_active_logs'] = combined_temp.groupby('interval_key')['geohash'].transform('count')
    
    combined_temp['interval_avg_temp'] = combined_temp.groupby('interval_key')['Temperature'].transform('mean')
    global_median_temp = combined_temp['Temperature'].median()
    combined_temp['interval_avg_temp'] = combined_temp['interval_avg_temp'].fillna(global_median_temp)
    
    print("Computing neighborhood activity spillover lags...")
    unique_ghs_list = combined_temp['geohash'].unique()
    coord_map = {gh: decode_geohash(gh) for gh in unique_ghs_list}
    train_unique_gh = combined_temp[['geohash']].drop_duplicates().reset_index(drop=True)
    train_unique_gh['latitude'] = train_unique_gh['geohash'].map(lambda x: coord_map[x][0])
    train_unique_gh['longitude'] = train_unique_gh['geohash'].map(lambda x: coord_map[x][1])
    coords_array = train_unique_gh[['latitude', 'longitude']].values
    
    nn = NearestNeighbors(n_neighbors=min(4, len(coords_array)), metric='euclidean')
    nn.fit(coords_array)
    _, indices = nn.kneighbors(coords_array)
    
    gh_list = train_unique_gh['geohash'].values
    neighbor_map = {}
    for i, gh in enumerate(gh_list):
        neighbor_map[gh] = [gh_list[idx] for idx in indices[i, 1:] if idx < len(gh_list)]
        
    for idx in range(3):
        combined_temp[f'neighbor_{idx}'] = combined_temp['geohash'].map(
            lambda x: neighbor_map.get(x, [x, x, x])[idx] if len(neighbor_map.get(x, [])) > idx else x
        )
        
    active_pairs = combined_temp[['interval_key', 'geohash']].drop_duplicates().copy()
    active_pairs['is_active'] = 1
    
    for idx in range(3):
        active_pairs_idx = active_pairs.rename(columns={'is_active': f'is_active_neigh_{idx}', 'geohash': f'geohash_neigh_{idx}'})
        combined_temp = combined_temp.merge(
            active_pairs_idx,
            left_on=['interval_key', f'neighbor_{idx}'],
            right_on=['interval_key', f'geohash_neigh_{idx}'],
            how='left'
        )
        combined_temp[f'is_active_neigh_{idx}'] = combined_temp[f'is_active_neigh_{idx}'].fillna(0)
        if f'geohash_neigh_{idx}' in combined_temp.columns:
            combined_temp = combined_temp.drop(columns=[f'geohash_neigh_{idx}'])
            
    combined_temp['neighbor_active_ratio'] = (
        combined_temp['is_active_neigh_0'] + 
        combined_temp['is_active_neigh_1'] + 
        combined_temp['is_active_neigh_2']
    ) / 3.0
    
    combined_temp = combined_temp.drop(columns=[
        'interval_key', 'neighbor_0', 'neighbor_1', 'neighbor_2',
        'is_active_neigh_0', 'is_active_neigh_1', 'is_active_neigh_2'
    ])
    
    train_clean = combined_temp.iloc[:len(train)].copy().reset_index(drop=True)
    test_clean = combined_temp.iloc[len(train):].copy().reset_index(drop=True)
    
    print("Computing spatial centralities...")
    train_clean, test_clean = compute_multiscale_spatial_centrality(train_clean, test_clean)
    
    print("Computing chronological features (AR and EWMA)...")
    for df in [train_clean, test_clean]:
        df['abs_minutes'] = (df['day'] - 1) * 1440 + df['minutes']
        
    train_clean['is_test'] = 0
    test_clean['is_test'] = 1
    train_clean['orig_index'] = train_clean.index
    test_clean['orig_index'] = test_clean.index
    
    combined_time = pd.concat([train_clean, test_clean], ignore_index=True)
    combined_time = combined_time.sort_values(by=['geohash', 'abs_minutes']).reset_index(drop=True)
    combined_time['time_delta_last_log'] = combined_time.groupby('geohash')['abs_minutes'].diff().fillna(1440.0)
    
    # DEMAND LAG 1
    combined_time['demand_lag1'] = combined_time.groupby('geohash')['demand'].shift(1).fillna(0.0)
    
    # EWMA
    combined_time['ewma_03'] = combined_time.groupby('geohash')['demand'].apply(lambda x: x.shift(1).ewm(alpha=0.3, min_periods=1).mean()).fillna(0.0).reset_index(drop=True)
    
    train_clean = combined_time[combined_time['is_test'] == 0].sort_values('orig_index').reset_index(drop=True)
    test_clean = combined_time[combined_time['is_test'] == 1].sort_values('orig_index').reset_index(drop=True)
    
    train_clean = train_clean.drop(columns=['is_test', 'orig_index'])
    test_clean = test_clean.drop(columns=['is_test', 'orig_index'])
    
    print("Computing OOF target encodings...")
    te_cols = ['Weather', 'RoadType', 'geohash_time_block', 'geohash_time_block_1h', 'geohash_weather']
    for col in te_cols:
        train_clean, test_clean = add_oof_target_encodings(train_clean, test_clean, col, 'idle_capacity')
        
    print("Computing OOF historical geohash stats...")
    train_clean, test_clean = add_oof_historical_geohash_stats(train_clean, test_clean)
    
    for df in [train_clean, test_clean]:
        df['LargeVehicles_bin'] = (df['LargeVehicles'] == 'Allowed').astype(int)
        df['Landmarks_bin'] = (df['Landmarks'] == 'Yes').astype(int)
        df['Bottleneck_Index'] = df['LargeVehicles_bin'] / df['NumberofLanes'].clip(lower=1)
        for col in ['Weather', 'RoadType']:
            df[col] = df[col].fillna('Missing')
            
    for col in ['NumberofLanes', 'Temperature']:
        median_val = train_clean[col].median()
        train_clean[col] = train_clean[col].fillna(median_val)
        test_clean[col] = test_clean[col].fillna(median_val)
        
    return train_clean, test_clean

---
## 5. Custom Asymmetric Loss Objectives
The penalty for under-predicting high-demand situations (e.g. `demand > 0.8`) is structurally much higher than over-predicting. 
We inject custom asymmetric MSE gradients directly into XGBoost and LightGBM. CatBoost operates natively on the Tweedie distribution.

In [ ]:
# Custom Asymmetric Objective Functions
def asym_mse_xgb(preds, dtrain):
    labels = dtrain.get_label()
    grad = preds - labels
    mask = (labels > 0.8) & (preds < labels)
    grad[mask] *= 15.0
    
    hess = np.ones_like(labels)
    hess[mask] *= 15.0
    return grad, hess

def asym_mse_lgb(labels, preds):
    grad = preds - labels
    mask = (labels > 0.8) & (preds < labels)
    grad[mask] *= 15.0
    
    hess = np.ones_like(labels)
    hess[mask] *= 15.0
    return grad, hess

---
## 6. Training & Auto-Regressive Decoding
We ensemble 3 robust algorithms: **CatBoost (Tweedie), LightGBM (Custom Asym), and XGBoost (Custom Asym)**.
Because the test set doesn't contain true past values for Day 49, we perform an Auto-Regressive loop: processing the test set timestamp-by-timestamp in absolute time order, continuously updating the `demand_lag1` and `ewma_03` features using our ensemble's own predictions.

In [ ]:
print("Loading data...")
train_path = '/home/aryank/Desktop/antigravity/train.csv'
test_path = '/home/aryank/Desktop/antigravity/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_processed, test_processed = pipeline_feature_engineering(train_df, test_df)

cat_features = ['geohash', 'day_of_week', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']

for col in cat_features:
    train_processed[col] = train_processed[col].astype(str)
    test_processed[col] = test_processed[col].astype(str)
    
numeric_features = [
    'day', 'is_weekend', 'time_block',
    'latitude', 'longitude',
    'spatial_centrality_3', 'spatial_centrality_10', 'spatial_centrality_20',
    'time_delta_last_log', 'demand_lag1', 'ewma_03',
    'minutes', 'sin_1440', 'cos_1440', 'sin_720', 'cos_720',
    'sin_480', 'cos_480', 'sin_360', 'cos_360',
    'NumberofLanes', 'LargeVehicles_bin', 'Landmarks_bin',
    'Bottleneck_Index', 'Temperature',
    'interval_active_logs', 'interval_avg_temp', 'neighbor_active_ratio',
    'Weather_te', 'RoadType_te', 'geohash_time_block_te',
    'geohash_mean_demand', 'geohash_std_demand', 'geohash_max_demand', 'geohash_spike_rate', 'geohash_time_block_spike_rate'
]
features = cat_features + numeric_features

train_processed_xgb = train_processed.copy()
test_processed_xgb = test_processed.copy()
for col in cat_features:
    train_processed_xgb[col] = train_processed_xgb[col].astype('category')
    test_processed_xgb[col] = test_processed_xgb[col].astype('category')
    
print(f"Total features: {len(features)}")

print("Training CatBoost (Tweedie)...")
cb_model1 = CatBoostRegressor(loss_function='Tweedie:variance_power=1.9', iterations=2000, learning_rate=0.025, depth=7, l2_leaf_reg=5.0, random_seed=42, verbose=False)
cb_model1.fit(train_processed[features], train_processed['idle_capacity'], cat_features=cat_features)

cb_model2 = CatBoostRegressor(loss_function='Tweedie:variance_power=1.9', iterations=2000, learning_rate=0.025, depth=7, l2_leaf_reg=5.0, random_seed=777, verbose=False)
cb_model2.fit(train_processed[features], train_processed['idle_capacity'], cat_features=cat_features)

print("Training LightGBM (Custom Asym)...")
lgb_model = LGBMRegressor(n_estimators=2400, learning_rate=0.012, num_leaves=63, random_state=42, n_jobs=-1, verbose=-1, objective=asym_mse_lgb)
lgb_model.fit(train_processed_xgb[features], train_processed_xgb['demand'])

print("Training XGBoost (Custom Asym)...")
dtrain = xgb.DMatrix(train_processed_xgb[features], label=train_processed_xgb['demand'], enable_categorical=True)
params = {'max_depth': 7, 'learning_rate': 0.015, 'tree_method': 'hist', 'seed': 42}
xgb_model = xgb.train(params, dtrain, num_boost_round=2000, obj=asym_mse_xgb)

print("Starting Auto-Regressive Decoding on Test Set...")
test_processed['timestamp_abs'] = test_processed['timestamp'].apply(lambda x: int(x.split(':')[0]) * 60 + int(x.split(':')[1]))
unique_timestamps = sorted(test_processed['timestamp_abs'].unique())

latest_demand = train_processed.groupby('geohash')['demand'].last().to_dict()

def ewma(series, alpha=0.3):
    return series.ewm(alpha=alpha, adjust=False).mean().iloc[-1]

latest_ewma = train_processed.groupby('geohash')['demand'].apply(ewma).to_dict()

predictions = np.zeros(len(test_processed))

for ts in unique_timestamps:
    mask = test_processed['timestamp_abs'] == ts
    if not mask.any(): continue
    
    current_ghs = test_processed.loc[mask, 'geohash']
    updated_lags = current_ghs.map(latest_demand).fillna(0.0)
    updated_ewma = current_ghs.map(latest_ewma).fillna(0.0)
    
    test_processed.loc[mask, 'demand_lag1'] = updated_lags
    test_processed_xgb.loc[mask, 'demand_lag1'] = updated_lags
    test_processed.loc[mask, 'ewma_03'] = updated_ewma
    test_processed_xgb.loc[mask, 'ewma_03'] = updated_ewma
    
    batch = test_processed.loc[mask, features]
    batch_xgb = test_processed_xgb.loc[mask, features]
    
    cb1_preds = 1.0 - cb_model1.predict(batch)
    cb2_preds = 1.0 - cb_model2.predict(batch)
    cb_preds = np.clip((cb1_preds + cb2_preds) / 2.0, 0.0, 1.0)
    
    lgb_preds = lgb_model.predict(batch_xgb)
    lgb_preds = np.clip(lgb_preds, 0.0, 1.0)
    
    dtest = xgb.DMatrix(batch_xgb, enable_categorical=True)
    xgb_preds = xgb_model.predict(dtest)
    xgb_preds = np.clip(xgb_preds, 0.0, 1.0)
    
    final_preds = (0.34 * cb_preds) + (0.33 * lgb_preds) + (0.33 * xgb_preds)
    predictions[mask] = final_preds
    
    # Update states for next iteration
    for gh, pred in zip(current_ghs, final_preds):
        latest_demand[gh] = pred
        latest_ewma[gh] = (1 - 0.3) * latest_ewma.get(gh, 0.0) + 0.3 * pred

test_processed['demand'] = predictions
output_path = '/home/aryank/Desktop/antigravity/final_solution.csv'
test_processed[['Index', 'demand']].to_csv(output_path, index=False)
print(f"Submission saved successfully to {output_path}!")